## Loading Fakeddit dataset

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "multimodal_only_samples" / "multimodal_train.tsv"

df = pd.read_csv(DATA_PATH, sep="\t")

print(df.shape)
print(df.columns)


(564000, 16)
Index(['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id',
       'image_url', 'linked_submission_id', 'num_comments', 'score',
       'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label',
       '6_way_label'],
      dtype='str')


## Initial Data Quality Assessment

In [2]:
df.head()

,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


In [3]:
print("\nData types:")
display(df.dtypes)

print("="*50)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("="*50)

print("\nDuplicate rows:", df.duplicated().sum())

print()
print("="*50)

print("\nImage availability:")
display(df["hasImage"].value_counts(dropna=False))

print("="*50)

print("\nTarget distribution:")
display(df["2_way_label"].value_counts(dropna=False))

print("="*50)

print("\nTarget proportions:")
display(df["2_way_label"].value_counts(normalize=True, dropna=False))


Data types:


author                      str
clean_title                 str
created_utc             float64
domain                      str
hasImage                   bool
id                          str
image_url                   str
linked_submission_id        str
num_comments            float64
score                     int64
subreddit                   str
title                       str
upvote_ratio            float64
2_way_label               int64
3_way_label               int64
6_way_label               int64
dtype: object


Missing values:


linked_submission_id    396143
domain                  167857
upvote_ratio            167857
num_comments            167857
author                   28710
image_url                 1534
created_utc                  0
clean_title                  0
hasImage                     0
id                           0
subreddit                    0
score                        0
title                        0
2_way_label                  0
3_way_label                  0
6_way_label                  0
dtype: int64


Duplicate rows: 0


Image availability:


hasImage
True    564000
Name: count, dtype: int64


Target distribution:


2_way_label
0    341919
1    222081
Name: count, dtype: int64


Target proportions:


2_way_label
0    0.606239
1    0.393761
Name: proportion, dtype: float64

In [4]:
display(
    df[
        [
            "id",
            "clean_title",
            "image_url",
            "hasImage",
            "2_way_label",
        ]
    ].head(10)
)

,id,clean_title,image_url,hasImage,2_way_label
0,awxhir,my walgreens offbrand mucinex was engraved wit...,https://external-preview.redd.it/WylDbZrnbvZdB...,True,1
1,98pbid,this concerned sink with a tiny hat,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,True,0
2,6f2cy5,hackers leak emails from uae ambassador to us,https://external-preview.redd.it/6fNhdbc6K1vFA...,True,1
3,4xypkv,puppy taking in the view,https://external-preview.redd.it/HLtVNhTR6wtYt...,True,1
4,8gnet9,i found a face in my sheet music too,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,True,0
5,2vkbtj,bride and groom exchange vows after fatal shoo...,https://external-preview.redd.it/FQ-J9OIPFRpqi...,True,1
6,6d50rl,major thermos,https://preview.redd.it/l9gvkkf3jizy.jpg?width...,True,0
7,86byl8,rabbi meat from cloned pig could be kosher for...,https://external-preview.redd.it/KHisCPOGwz7cz...,True,1
8,cgp0lmq,cutouts,http://i.imgur.com/vbveIEd%2ejpg,True,0
9,98vrcq,jesus christ converting local teens to christi...,https://preview.redd.it/31aw9wjucah11.jpg?widt...,True,0


## Split Integrity and Distribution Analysis

In [5]:
from pathlib import Path

import pandas as pd

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "multimodal_only_samples"
)

TRAIN_PATH = DATA_DIR / "multimodal_train.tsv"
VALIDATE_PATH = DATA_DIR / "multimodal_validate.tsv"
TEST_PATH = DATA_DIR / "multimodal_test_public.tsv"

train_df = pd.read_csv(TRAIN_PATH, sep="\t")
validate_df = pd.read_csv(VALIDATE_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")

splits = {
    "train": train_df,
    "validate": validate_df,
    "test": test_df,
}

for name, split in splits.items():
    print("="*50)
    print(f"{name:10s}: {split.shape}")
    print(split["2_way_label"].value_counts(normalize=True).sort_index())
    print()

train     : (564000, 16)
2_way_label
0    0.606239
1    0.393761
Name: proportion, dtype: float64

validate  : (59342, 16)
2_way_label
0    0.607024
1    0.392976
Name: proportion, dtype: float64

test      : (59319, 16)
2_way_label
0    0.603719
1    0.396281
Name: proportion, dtype: float64



In [6]:
# Chekcing id overlap across splits

train_ids = set(train_df["id"])
validate_ids = set(validate_df["id"])
test_ids = set(test_df["id"])

print("Train ∩ Validate:", len(train_ids & validate_ids))
print("Train ∩ Test:    ", len(train_ids & test_ids))
print("Validate ∩ Test: ", len(validate_ids & test_ids))

Train ∩ Validate: 0
Train ∩ Test:     0
Validate ∩ Test:  0


In [7]:
# Checking image availability across splits

for name, split in splits.items():
    missing_images = split["image_url"].isna().sum()
    empty_titles = split["clean_title"].isna().sum()

    print(
        f"{name:10s} | "
        f"missing image_url: {missing_images:6d} | "
        f"missing clean_title: {empty_titles:4d}"
    )

train      | missing image_url:   1534 | missing clean_title:    0
validate   | missing image_url:    173 | missing clean_title:    0
test       | missing image_url:    156 | missing clean_title:    0


## Image URL Overlap Check

In [8]:
candidate_splits = {
    "train": pd.read_csv(
        PROJECT_ROOT / "data" / "processed" / "train_candidates.csv"
    ),
    "validate": pd.read_csv(
        PROJECT_ROOT / "data" / "processed" / "validate_candidates.csv"
    ),
    "test": pd.read_csv(
        PROJECT_ROOT / "data" / "processed" / "test_candidates.csv"
    ),
}

train_urls = set(candidate_splits["train"]["image_url"].dropna())
validate_urls = set(candidate_splits["validate"]["image_url"].dropna())
test_urls = set(candidate_splits["test"]["image_url"].dropna())

print("Train ∩ Validate:", len(train_urls & validate_urls))
print("Train ∩ Test:    ", len(train_urls & test_urls))
print("Validate ∩ Test: ", len(validate_urls & test_urls))

Train ∩ Validate: 8
Train ∩ Test:     9
Validate ∩ Test:  3


In [9]:
from pathlib import Path
import pandas as pd

processed_dir = PROJECT_ROOT / "data" / "processed"

train = pd.read_csv(processed_dir / "train.csv")

display(train.head())

print("Example image path:")
print(train.loc[0, "image_path"])

print("\nAll image paths are relative:")
print(
    all(
        not Path(path).is_absolute()
        for path in train["image_path"]
    )
)

,id,text,image_path,label,image_url,sha256,phash,width,height
0,4jj0fh,funny dog names you havent thought of,images/train/4jj0fh.jpg,0,https://external-preview.redd.it/9oKz4f9RB8AKo...,e05609ffbae969fd6211e8f14970ebe9d08b2c4558c010...,a6d879265ac12f27,320.0,240.0
1,7tykik,destyler numb,images/train/7tykik.jpg,0,https://preview.redd.it/9ty8imebx4d01.jpg?widt...,78982719121bfb2417ffdc7f88e2c04a29498a5810e87b...,b0dfcb3164c7911c,320.0,320.0
2,1jfqnp,us student abandoned in san diego jail gets m ...,images/train/1jfqnp.jpg,1,https://external-preview.redd.it/dWFkSwgTS4BHi...,9800dfe906502c6693c8fb1a10149a8e3eaee9aa74dc80...,f44333d38fcc8cc8,320.0,180.0
3,boa5ua,this horrified styrofoam,images/train/boa5ua.jpg,0,https://preview.redd.it/3tc1q8tzz1y21.jpg?widt...,4352e1b262dd177e03a3cb386e8bee2611f9c13a57255e...,85957aa53e6a3878,320.0,426.0
4,1aco8p,je crois i believe canadian wwii poster x,images/train/1aco8p.jpg,0,https://external-preview.redd.it/Xx0LF7w2FhzyX...,46e743177f80438597dcc6d59a728bfb587782bd35860f...,e69b9e2c89963869,320.0,430.0


Example image path:
images/train/4jj0fh.jpg

All image paths are relative:
True
